# SleepStageNet CNN+BiLSTM — Colab Training

Run 20-fold LOSO-CV on Colab T4 GPU.

**Runtime:** Select **GPU (T4)** kernel before running: Select Kernel → Colab → New Colab Server → GPU → T4

In [14]:
# 1. Configuration — edit these paths/settings as needed
import os

# Where to clone the repo (Colab ephemeral storage is fine — code is lightweight)
REPO_DIR = "/content/deepsleepnet-lite"

# OUTPUT_DIR points to Google Drive — not Colab's ephemeral /content storage.
# This is critical: Colab sessions disconnect after ~12 hours (or sooner on free tier).
# With output on Drive, completed folds survive crashes. Combined with --skip_existing,
# you can simply reconnect and re-run the training cell to resume from where it left off.
OUTPUT_DIR = "/content/drive/MyDrive/University-of-Washington/Winter-2026/SleepStageNet/temporal_output"

# Training hyperparameters
SEQ_LEN = 5         # Sequence length (should be odd: 3, 5, 11, 21)
BATCH_SIZE = 64      # Batch size
FOLDS = range(20)    # Which folds to train (range(20) for full LOSO-CV)

# Data download (Google Drive file ID for the preprocessed Sleep-EDF zip)
DATA_FILE_ID = "1wDu9tl6_P250k522tQC9LUUVh7ocG1_x"
DATA_DIR = f"{REPO_DIR}/data/SleepEDF/processed/eeg_FpzCz_PzOz_v1"

# Repo URL
REPO_URL = "https://github.com/manishdas/deepsleepnet-lite.git"

In [15]:
# 2. Mount Drive, install deps, clone/update repo
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-learn gdown

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")
    if input("Pull latest changes? (y/n): ").strip().lower() == 'y':
        !cd {REPO_DIR} && git pull

print(f"Output dir: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists at /content/deepsleepnet-lite
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 23 (delta 5), reused 23 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 3.92 MiB | 14.08 MiB/s, done.
From https://github.com/manishdas/deepsleepnet-lite
   3b91e15..daf6277  main       -> origin/main
Updating 3b91e15..daf6277
Fast-forward
 docs/team_handoff.md                          |  54 ++-
 temporal/README.md                            | 153 ++-----
 temporal/SleepStageNet_CNN_BiLSTM_Colab.ipynb |  90 ++++
 temporal/figures/class_distribution.png       | Bin 0 -> 53986 bytes
 temporal/figures/confusion_matrices.png       | Bin 0 -> 92963 bytes
 temporal/figures/sample_eeg_epoch.png         | Bin 0 -> 103343 bytes
 temporal/figures/training_

In [16]:
# 3. Download data (skip if already exists)
import glob

eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))

if len(eeg_files) >= 39:
    print(f"Data already exists: {len(eeg_files)} files")
else:
    !gdown {DATA_FILE_ID} -O /tmp/data.zip
    !cd {REPO_DIR} && unzip -q -o /tmp/data.zip
    eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))
    print(f"Downloaded: {len(eeg_files)} files")

Data already exists: 39 files


In [17]:
# 4. Verify GPU
import torch, multiprocessing
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'CPU cores: {multiprocessing.cpu_count()}')

CUDA: True
Device: Tesla T4
CPU cores: 2


In [18]:
# 5. Sanity check — 2 epochs per stage (quick validation before full run)
%cd {REPO_DIR}/temporal
!python train_sequence.py --fold 0 --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
    --cnn_epochs 2 --lstm_epochs 2 --finetune_epochs 2 \
    --output_dir {OUTPUT_DIR}/sanity_check

/content/deepsleepnet-lite/temporal
Logging to: /content/drive/MyDrive/University-of-Washington/Winter-2026/SleepStageNet/temporal_output/sanity_check/train_fold0_L5.log

Fold 0 | seq_length=5 | device=cuda
CPU cores: 2 | DataLoader workers: 1 | PyTorch threads: 2

[1] Loading data...
  Train: 32725 sequences
  Val:   7467 sequences
  Test:  1960 sequences
  Class weights: [0.963 3.162 0.489 1.449 1.098]

[2] Stage 1: Pre-training CNN on individual epochs...
  [CNN pretrain] epoch   1/2: train loss=0.6866 acc=0.748 f1=0.694 | val loss=0.6037 acc=0.783 f1=0.754 kappa=0.696 lr=1.0e-03 (9.2s) *
  Saved CNN weights: /content/drive/MyDrive/University-of-Washington/Winter-2026/SleepStageNet/temporal_output/sanity_check/cnn_fold0.pt
  CNN-only val: acc=0.783 f1_macro=0.754 kappa=0.696

  Loaded pre-trained CNN into sequence model

[3] Stage 2: Training BiLSTM (CNN frozen)...
  Trainable params: 199,941
  [LSTM] epoch   1/2: train loss=0.4678 acc=0.837 f1=0.793 | val loss=0.4847 acc=0.809 f1=0

In [ ]:
# 6. Full training (saves directly to Drive, skip_existing for crash recovery)
for fold in FOLDS:
    !python train_sequence.py --fold {fold} --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
        --output_dir {OUTPUT_DIR} --skip_existing

Logging to: /content/drive/MyDrive/University-of-Washington/Winter-2026/SleepStageNet/temporal_output/train_fold0_L5.log

Fold 0 | seq_length=5 | device=cuda
CPU cores: 2 | DataLoader workers: 1 | PyTorch threads: 2

[1] Loading data...
  Train: 32725 sequences
  Val:   7467 sequences
  Test:  1960 sequences
  Class weights: [0.963 3.162 0.489 1.449 1.098]

[2] Stage 1: Pre-training CNN on individual epochs...
  [CNN pretrain] epoch   1/50: train loss=0.6741 acc=0.753 f1=0.697 | val loss=0.5916 acc=0.796 f1=0.755 kappa=0.711 lr=1.0e-03 (8.9s) *
  [CNN pretrain] epoch   3/50: train loss=0.5320 acc=0.808 f1=0.760 | val loss=0.5527 acc=0.808 f1=0.766 kappa=0.725 lr=1.0e-03 (7.7s) *
  [CNN pretrain] epoch   5/50: train loss=0.5088 acc=0.814 f1=0.768 | val loss=0.5586 acc=0.783 f1=0.756 kappa=0.700 lr=1.0e-03 (8.0s)
  [CNN pretrain] epoch   7/50: train loss=0.4872 acc=0.823 f1=0.779 | val loss=0.5409 acc=0.817 f1=0.780 kappa=0.740 lr=1.0e-03 (8.3s) *
  [CNN pretrain] epoch  10/50: train los

In [ ]:
# 7. Verify results on Drive
!ls {OUTPUT_DIR}/fold*_results.json | wc -l
!ls {OUTPUT_DIR}/*.pt | wc -l

In [ ]:
# 8. Generate plots (saved to Drive for persistence)
FIGURES_DIR = f"{OUTPUT_DIR}/figures"

!python plot_results.py --mode all \
    --results_dir {OUTPUT_DIR} \
    --output_dir {FIGURES_DIR} \
    --data_dir {DATA_DIR}

# Show the generated figures inline
from IPython.display import Image, display
import glob
for fig in sorted(glob.glob(f'{FIGURES_DIR}/*.png')):
    print(f'\n{os.path.basename(fig)}')
    display(Image(filename=fig))

In [ ]:
# 9. Summary — aggregate metrics across all folds
import json, glob, numpy as np

results_files = sorted(glob.glob(f'{OUTPUT_DIR}/fold*_results.json'))
print(f'Completed folds: {len(results_files)} / {len(list(FOLDS))}')

if results_files:
    accs, f1s, kappas = [], [], []
    for fp in results_files:
        with open(fp) as f:
            r = json.load(f)
        accs.append(r['test_metrics']['accuracy'])
        f1s.append(r['test_metrics']['f1_macro'])
        kappas.append(r['test_metrics']['kappa'])
    print(f'\nTest accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}')
    print(f'Test F1 macro:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    print(f'Test kappa:     {np.mean(kappas):.4f} ± {np.std(kappas):.4f}')